# Playground Series - Season 6 Episode 8

### Imports

In [ ]:
import kagglehub
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score

from catboost import CatBoostClassifier

### Config

In [ ]:
SEED = 42
N_SPLITS = 5 # number of outer CV folds
ES_SIZE = 0.1 # share of each folds train part reserved for early stopping
TARGET = 'addicted_label'

In [ ]:
PARAMS = dict(
    iterations=15_000,
    learning_rate=0.03,
    random_seed=SEED,
    logging_level='Silent',
    eval_metric='AUC',
    early_stopping_rounds=100
)

### Load Data

In [ ]:
COMP = 'playground-series-s6e8'
PATH = kagglehub.competition_download(COMP)

train = pd.read_csv(f'{PATH}/train.csv', index_col='id')
test = pd.read_csv(f'{PATH}/test.csv', index_col='id')

### Feature Engineering

In [ ]:
def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    data = df.drop(columns=['gender', 'stress_level', 'academic_work_impact'])

    daily = data['daily_screen_time_hours']
    gaming = data['gaming_hours']
    work = data['work_study_hours']
    weekend = data['weekend_screen_time']

    # min_count=3 -> NaN unless all three activities are reported, so a partial sum is never mistaken for a complete one
    logged = data[['social_media_hours', 'gaming_hours', 'work_study_hours']].sum(axis=1, min_count=3)
    data['unlogged_hours'] = daily - logged

    # Best available daily estimate: reported -> sum of activities -> rescaled weekend
    # Median of weekend_screen_time / daily_screen_time_hours on train (1.2407) is used to back out a daily estimate when the daily value is missing

    daily_hat = daily.fillna(logged).fillna(weekend / 1.24)
    data['daily_screen_time_filled'] = daily_hat

    # Upper bound on social media time implied by the reported total
    # Uses raw daily, not daily_hat: an imputed total would turn this bound into a guess about a guess
    data['social_upper_bound'] = daily - (gaming + work)

    return data

In [ ]:
X, y = feature_engineering(train.drop(columns=[TARGET])), train[TARGET]
X_test = feature_engineering(test)

### Validation Setup

In [ ]:
# One fixed split, reused by every later experiment so the numbers stay comparable
folds = list(StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED).split(X, y))

### Model Training

In [ ]:
oof = np.zeros(len(X))
models = []
scores = []

for fold, (train_idx, val_idx) in enumerate(folds):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    # The early-stopping set is carved out of the folds training part only, so it never overlaps with the folds scoring set
    X_fit, X_es, y_fit, y_es = train_test_split(
        X_train, y_train, test_size=ES_SIZE, stratify=y_train, random_state=SEED + fold
    )

    model = CatBoostClassifier(**PARAMS)
    model.fit(X_fit, y_fit, eval_set=(X_es, y_es))

    oof[val_idx] = model.predict_proba(X_val)[:, 1]
    models.append(model)
    
    score = roc_auc_score(y_val, oof[val_idx])
    scores.append(score)
    print(f'split{fold}_test_score: {score:.5f} (best_iteration: {model.get_best_iteration():,})')

print()
print(f'cv_scores: {scores}')
print(f'avg_score: {np.mean(scores):.5f} (std_dev: {np.std(scores):.5f})')
print(f'pooled_OOF: {roc_auc_score(y, oof):.5f}')

### Submission

In [ ]:
# Every fold model predicts the test set; their mean is more stable than any single one
test_pred = np.mean([m.predict_proba(X_test)[:, 1] for m in models], axis=0)

submission = pd.DataFrame({TARGET: test_pred}, index=X_test.index)
submission.to_csv('data/submission.csv')